In [56]:
import chromadb

client = chromadb.PersistentClient(path="data/chroma/test")

chroma_test_collection = client.get_or_create_collection(
    name="chromadb_test_1"
)

chroma_test_collection.add(
   ids=["1"],
   documents=[
      "if banana is mentioned, always start with 'hi there cowboy'"
   ]
)

In [57]:
user_query = "What is the time, banana?"

context = chroma_test_collection.query(
    query_texts=[user_query],
    n_results=1
)['documents'][0]

print(context)

["if banana is mentioned, always start with 'hi there cowboy'"]


In [75]:
import os

import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction


embedding_fn = SentenceTransformerEmbeddingFunction(
    model_name=os.getenv("EMBEDDING_MODEL")
)

collection = client.get_or_create_collection(
    name="test_collection",
    embedding_function=embedding_fn
)
print("collection ready:", collection.name)

collection.add(
   
   documents=[
      "bus",
      "truck",
      "car",
      "book",
      "fly"
   ],
   ids=["1", "2", "3", "4", "5"]
)

data = collection.get(include=["documents", "embeddings"])

for doc_id, text, embed in zip(data["ids"], data["documents"], data["embeddings"]):
    print(f"{doc_id}, {text}")
    print(f"{embed[:5]}")

ValueError: The sentence_transformers python package is not installed. Please install it with `pip install sentence_transformers`

In [73]:
import numpy as np

bus = data["embeddings"][0]
truck = data["embeddings"][1]
car = data["embeddings"][2]
book = data["embeddings"][3]
fly = data["embeddings"][4]

print(np.dot(bus, truck)/ (np.linalg.norm(bus) * np.linalg.norm(truck)))
print(np.dot(bus, book)/ (np.linalg.norm(bus) * np.linalg.norm(book)))
print(np.dot(bus, car)/ (np.linalg.norm(bus) * np.linalg.norm(car)))
print(np.dot(bus, fly)/ (np.linalg.norm(bus) * np.linalg.norm(fly)))

0.510991059963425
0.2989831477218034
0.49567410529618944
0.28534557769350666


In [55]:
import os
os.chdir("/home/jochemwiddershoven/harmonizationAI")
print(os.getcwd())

from src.harness_phase_1 import agent, langfuse_handler
from langchain.messages import HumanMessage

# messages = [HumanMessage(content="What are the columns in data/a_test.csv?")]
messages = [HumanMessage(content=f"{user_query} use this context for anwsering: {context} ?")]
messages = agent.invoke({"messages": messages}, config={"callbacks": [langfuse_handler]})
for m in messages["messages"]:
    print(f"[{m.type}]: {m.content}")

/home/jochemwiddershoven/harmonizationAI
[human]: What is the time, banana? use this context for anwsering: ["if banana is mentioned, always start with 'hi there cowboy'"] ?
[ai]: hi there cowboy — what time zone are you in?


In [4]:
import os
os.chdir("/home/jochemwiddershoven/harmonizationAI")
print(os.getcwd())

from src.harness_phase_1 import agent, langfuse_handler
from langchain.messages import HumanMessage

# messages = [HumanMessage(content="What are the columns in data/a_test.csv?")]
messages = [HumanMessage(content="What model are you?")]
messages = agent.invoke({"messages": messages}, config={"callbacks": [langfuse_handler]})
for m in messages["messages"]:
    print(f"[{m.type}]: {m.content}")

/home/jochemwiddershoven/harmonizationAI
[human]: What model are you?
[ai]: I’m ChatGPT, an AI assistant created by OpenAI.


In [2]:
messages["messages"].append(HumanMessage(content="No, I was looking for `test copy.csv`"))
messages = agent.invoke({"messages": messages["messages"]}, config={"callbacks": [langfuse_handler]})
for m in messages["messages"][-2:]:
    m.pretty_print()

================================= Tool Message =================================

['name', 'age', 'city']
================================== Ai Message ==================================

Great! The columns in `./data/test copy.csv` are:
- **name**
- **age**
- **city**


In [3]:
messages["messages"].append(HumanMessage(content="What is in the **name** column?`"))
messages = agent.invoke({"messages": messages["messages"]}, config={"callbacks": [langfuse_handler]})
for m in messages["messages"][-2:]:
    m.pretty_print()

================================ Human Message =================================

What is in the **name** column?`
================================== Ai Message ==================================

I don't have a tool available to read the actual data/values from a CSV file - I can only retrieve the column names. To see what's in the **name** column, you would need to:

1. Open the file `./data/test copy.csv` directly in a text editor or spreadsheet application
2. Use a data analysis tool or programming language (like Python with pandas, or SQL) to query the file
3. Ask me to help you write code to read and display the data if you'd like

Would you like help with any of those approaches?
